In [1]:
# import libraries
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import joblib

In [2]:
df = pd.read_csv('housing_cleaned.csv')

In [3]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,household_rooms
0,-122.23,37.88,41.0,6.781058,4.867534,5.777652,4.844187,8.3252,452600.0,NEAR BAY,1.399834
1,-122.22,37.86,21.0,8.867850,7.009409,7.784057,7.037906,8.3014,358500.0,NEAR BAY,1.260013
2,-122.24,37.85,52.0,7.291656,5.252273,6.208590,5.181784,7.2574,352100.0,NEAR BAY,1.407171
3,-122.25,37.85,52.0,7.150701,5.463832,6.326149,5.393628,5.6431,341300.0,NEAR BAY,1.325768
4,-122.25,37.85,52.0,7.395108,5.638355,6.338594,5.560682,3.8462,342200.0,NEAR BAY,1.329892
...,...,...,...,...,...,...,...,...,...,...,...
20425,-121.09,39.48,25.0,7.418181,5.926926,6.740519,5.802118,1.5603,78100.0,INLAND,1.278530
20426,-121.21,39.49,18.0,6.548219,5.017280,5.877736,4.744932,2.5568,77100.0,INLAND,1.380045
20427,-121.22,39.43,17.0,7.720905,6.186209,6.915723,6.073045,1.7000,92300.0,INLAND,1.271340
20428,-121.32,39.43,18.0,7.528869,6.016157,6.609349,5.857933,1.8672,84700.0,INLAND,1.285243


# Feature-Target Splitting
We separate the dataset into X and Y to distinguish between the data we learn from and the value we want to predict.

In [4]:
x = df.drop(['median_house_value'],axis = 1)
y = df['median_house_value']

# Train-Test Splitting
We split the data into 4 subsets to train the model and then evaluate its performance on unseen data.

In [5]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)

train_data = x_train.join(y_train)
test_data = x_test.join(y_test)

- (X_train, y_train) used to train the model.
- (X_test, y_test) used to "evaluate" the model.
- (train_data, test_data) used for applying transformations each one separately 

# Transformation After Splitting

Encoding Transformation For "ocean_proximity"

In [6]:
train_data = train_data.join(pd.get_dummies(train_data.ocean_proximity,dtype= int)).drop(['ocean_proximity'],axis = 1)
test_data = test_data.join(pd.get_dummies(test_data.ocean_proximity,dtype= int)).drop(['ocean_proximity'],axis = 1)

# Avoiding expected error 
test_data = test_data.reindex(columns = train_data.columns, fill_value=0)


Check the result

In [7]:
train_data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,household_rooms,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
14141,-117.12,32.71,33.0,7.136483,5.805135,7.182352,5.774552,1.9286,1.235851,78500.0,0,0,0,0,1
13110,-117.66,34.15,20.0,7.833996,5.743003,6.873164,5.655992,8.0103,1.385079,395500.0,0,1,0,0,0
775,-122.10,37.61,35.0,7.767264,6.129050,7.454720,6.148468,4.5281,1.263284,173600.0,0,0,0,1,0
18515,-122.39,40.58,44.0,7.393878,5.973810,6.851185,5.852202,1.5972,1.263435,68900.0,0,1,0,0,0
5116,-118.27,33.94,30.0,6.948897,5.620401,6.777647,5.602119,1.5268,1.240405,91600.0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4809,-118.27,34.03,50.0,5.981414,5.451038,6.855409,5.497168,1.7546,1.088090,175000.0,1,0,0,0,0
9296,-122.53,37.91,37.0,7.833996,5.988961,6.907755,6.035481,7.9892,1.297990,500001.0,0,0,0,1,0
4497,-118.20,34.03,52.0,6.652863,5.347108,6.701960,5.318120,2.3472,1.250980,135200.0,1,0,0,0,0
11247,-117.92,33.74,24.0,8.579604,6.969791,8.297045,6.954639,4.3882,1.233652,189300.0,1,0,0,0,0


In [8]:
test_data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,household_rooms,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
14888,-117.00,32.79,31.0,8.088562,6.588926,7.375256,6.561031,3.4773,1.232819,155800.0,1,0,0,0,0
6171,-117.97,34.05,33.0,7.281386,5.594711,7.150701,5.631212,3.6563,1.293041,162700.0,1,0,0,0,0
7968,-118.19,33.85,30.0,8.170186,6.967909,7.893199,6.941190,2.2417,1.177058,151900.0,0,0,0,0,1
2000,-119.83,36.73,21.0,7.440147,5.883322,7.206377,5.758902,2.4137,1.291938,62100.0,0,1,0,0,0
3381,-118.31,34.26,37.0,7.275865,5.509388,6.437752,5.480639,5.7600,1.327558,239400.0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10699,-117.93,33.64,31.0,7.163947,5.877736,7.133296,5.924256,2.7143,1.209257,185400.0,1,0,0,0,0
14814,-116.95,32.76,13.0,8.620472,6.754604,7.637716,6.603944,4.9528,1.305352,266200.0,1,0,0,0,0
1258,-121.71,37.99,27.0,8.258940,6.577861,7.643004,6.562444,3.3558,1.258516,129700.0,0,1,0,0,0
10408,-117.71,33.52,17.0,7.818832,6.035481,6.776507,5.891644,6.1007,1.327105,340900.0,1,0,0,0,0


# Training The Model

In [9]:
x_train, y_train = train_data.drop(['median_house_value'], axis= 1), train_data['median_house_value']
x_test, y_test = test_data.drop(['median_house_value'], axis= 1), test_data['median_house_value']

In [10]:
reg = LinearRegression()

reg.fit(x_train,y_train)

LinearRegression()

In [11]:
reg.score(x_test,y_test)

0.6708144303417011

In [12]:
forest = RandomForestRegressor()

forest.fit(x_train,y_train)

RandomForestRegressor()

In [13]:
forest.score(x_test,y_test)

0.8196226599842884

# Pickle the model
Saving the Model

In [14]:
joblib.dump(forest, 'model.pkl')

['model.pkl']